# ※ 과제 안내
- 과제 배점: 각 문제당 10점, 총점 100점입니다. 부분 점수는 제공되지 않습니다.  

- 채점 기준:  
    - 출력 결과 일치: 제출한 코드가 제시된 출력 결과와 일치하는 경우에만 정답으로 인정됩니다.  
    - 코드의 다양성 인정: 출력 결과가 동일하다면 다양한 접근 방식을 존중하여 정답으로 인정합니다.

---
# 14주차 과제

---
### Finetuning 과제

In [5]:
# 실습 환경 세팅

!pip install -q transformers datasets peft torch

In [6]:
# 공통 실행 환경 (모든 문제 공통))

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

In [7]:
# 개인이 발급받은 huggingface token 사용해주세요!
# 사용하고자 하는 모델에 대해서는 huggingface model card에서 access를 요청하셔야 사용가능합니다 (최초 1회만 필요)

from huggingface_hub import notebook_login

notebook_login()

In [8]:
# 문제 1. pretrained model & baseline 추론 & Instruction Dataset 구성 및 전처리

# 문제 1-1. Tokenizer 및 모델 로드
# 사전학습 언어모델(Llama-3.2-1B-Instruct)과 토크나이저를 로드하시오.

# TODO
model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.
401 Client Error. (Request ID: Root=1-69bb97c3-5a2a126c60d698d700d369de;3d8ea79d-218b-4aeb-9cea-5cb318aeaa3e)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# 문제 1-2. Fine-tuning 이전 Baseline 추론
# Fine-tuning 이전 모델의 응답을 생성하시오.

prompt = "Explain fine-tuning in simple terms."
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    # TODO
    outputs = model.generate(**inputs, max_length=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# 문제 1-3. Instruction / Response 포맷 구성

dataset = load_dataset("tatsu-lab/alpaca", split="train[:200]")

def format_example(example):
    # TODO
    text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    return {"text": text}

In [ ]:
# 문제 1-4. Tokenization + Label 생성
# Causal LM 학습을 위한 전처리를 완성하시오.

def preprocess(example):
    # TODO
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    # TODO
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(
    preprocess,
    remove_columns=dataset.column_names
)

In [ ]:
# 문제 2. LoRA 기반 Fine-tuning 구성
# 문제 2-1. LoRA 설정 정의
# 다음 조건을 만족하도록 LoRA 설정을 구성하시오.

# rank = 8
# attention projection(q_proj, v_proj)에만 적용
# dropout 사용


# TODO
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
# 문제 2-2. LoRA 적용 및 Trainable Parameter 확인

# TODO
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()


In [ ]:
# 문제 2-3. Trainer 구성
# 다음 조건을 만족하도록 Trainer를 구성하시오.

# batch size = 2
# epoch = 1
# checkpoint 저장 없음
# 외부 로깅 없음

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# TODO
training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    save_strategy="no",
    report_to="none"
)

# TODO
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)


In [ ]:
# 문제 2-4. Fine-tuning 실행

# TODO
trainer.train()

In [ ]:
# 문제 3. Fine-tuning 결과 추론 및 모델 재사용
# 문제 3-1. Fine-tuned 모델 추론

prompt = "### Instruction:\nExplain LoRA.\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt")

device = lora_model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    # TODO
    outputs = lora_model.generate(
      **inputs,
      max_length=80,
      do_sample=True
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
# 문제 3-2. LoRA 모델 저장

save_path = "./final_lora_model"

# TODO
lora_model.save_pretrained(save_path)



In [ ]:
# 문제 3-3. 모델 재로드 후 추론

# TODO
base_model = AutoModelForCausalLM.from_pretrained(model_name)
reloaded_model = get_peft_model(base_model, lora_config)
reloaded_model.load_state_dict(lora_model.state_dict())

device = inputs["input_ids"].device
reloaded_model = reloaded_model.to(device)

with torch.no_grad():
    outputs = reloaded_model.generate(**inputs, max_length=80)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


In [ ]:
# 문제 4. Full Fine-Tuning vs LoRA 비교
# 문제 4-1. Full Fine-Tuning 모델 파라미터 수 계산

# TODO
full_ft_model = AutoModelForCausalLM.from_pretrained(model_name)

device = inputs["input_ids"].device
full_ft_model = full_ft_model.to(device)

total_params = sum(p.numel() for p in full_ft_model.parameters())
trainable_params = sum(
    p.numel() for p in full_ft_model.parameters() if p.requires_grad
)

print(trainable_params, total_params)


In [ ]:
# 문제 4-2. LoRA 모델 파라미터 수 계산

# TODO
lora_total = sum(p.numel() for p in lora_model.parameters())
lora_trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)


In [ ]:
# 문제 4-3. 동일 프롬프트 추론 비교

# TODO
with torch.no_grad():
    out_full = full_ft_model.generate(**inputs, max_length=80)
    out_lora = lora_model.generate(**inputs, max_length=80)



In [ ]:
# 문제 5. Multi-LoRA & Forgetting 완화
# 문제 5-1. Task A용 LoRA Adapter 생성

# TODO
lora_config_A = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# TODO
base_model_A = AutoModelForCausalLM.from_pretrained(model_name)
model_A = get_peft_model(base_model_A, lora_config_A)


In [ ]:
# 문제 5-2. Task B용 LoRA Adapter 생성

# TODO
lora_config_B = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

# TODO
base_model_B = AutoModelForCausalLM.from_pretrained(model_name)
model_B = get_peft_model(base_model_B, lora_config_B)


In [ ]:
# 문제 5-3. Adapter 교체 기반 추론

prompt = "Explain fine-tuning briefly."
inputs = tokenizer(prompt, return_tensors="pt")

# TODO
with torch.no_grad():
    out_A = model_A.generate(**inputs, max_length=50)
    out_B = model_B.generate(**inputs, max_length=50)



---
### Multimodal LLM 과제

In [ ]:
! pip install -q transformers torch

In [ ]:
# 공통 실습 환경 (모든 문제 공통)

import torch
import torch.nn as nn
from PIL import Image

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    CLIPVisionModel,
    CLIPImageProcessor
)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 문제 6

# 다음 조건을 만족하도록 코드를 완성하시오.
# 입력 이미지를 CLIP processor로 변환
# Vision Encoder forward 수행
# 출력 tensor의 shape 출력
# GPT 계열 tokenizer를 사용해 padding 문제 없이 text를 tokenization 하시오.

image = Image.new("RGB", (224, 224), color="white")

processor = CLIPImageProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)
vision_model = CLIPVisionModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

# TODO 1: image → tensor 변환
inputs = processor(
    images=image,
    return_tensors="pt"
).to(device)

# TODO 2: vision encoder forward
with torch.no_grad():
    outputs = vision_model(**inputs)

vision_embeds = outputs.last_hidden_state
print("Vision embedding shape:", vision_embeds.shape)


tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Describe the image in detail."

# TODO 3: padding token 설정
tokenizer.pad_token = tokenizer.eos_token

# TODO 4: tokenizer 적용
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True
)

print(inputs["input_ids"].shape)

In [ ]:
# 문제 7

# Vision embedding을 language hidden size로 변환하는 projection layer를 정의하시오.

vision_dim = 768
language_dim = 768

# TODO 1: projection layer 정의
vision_to_text = nn.Linear(vision_dim, language_dim)

dummy_vision = torch.randn(1, 10, vision_dim)
projected = vision_to_text(dummy_vision)

print(projected.shape)


In [ ]:
# 문제 8

# Vision embedding과 Text embedding을 sequence dimension으로 concat하시오.

batch_size = 1
vision_tokens = 10
text_tokens = 6
hidden_dim = 768

vision_embeds = torch.randn(batch_size, vision_tokens, hidden_dim)
text_embeds = torch.randn(batch_size, text_tokens, hidden_dim)

# TODO: concat 수행
multimodal_embeds = torch.cat(
    [vision_embeds, text_embeds],
    dim=1
)

print(multimodal_embeds.shape)


In [ ]:
# 문제 9

# 아래 조건을 만족하는 forward()를 완성하시오.

# CLIP Vision Encoder 사용
# Vision → Language projection
# Text embedding과 concat
# Language Model에 inputs_embeds 전달

class MiniLLaVAModel(nn.Module):
    def __init__(self, vision_model, lm, tokenizer):
        super().__init__()
        self.vision_model = vision_model
        self.lm = lm
        self.tokenizer = tokenizer
        self.proj = nn.Linear(768, lm.config.hidden_size)

    def forward(self, pixel_values, input_ids, labels=None):
        # TODO 1: vision embedding
        vision_outputs = self.vision_model(pixel_values=pixel_values)

        # TODO 2: projection
        vision_embeds = self.proj(vision_outputs.last_hidden_state)

        # TODO 3: text embedding
        text_embeds = self.lm.get_input_embeddings()(input_ids)

        # TODO 4: concat
        inputs_embeds = torch.cat([vision_embeds, text_embeds], dim=1)

        return self.lm(
            inputs_embeds=inputs_embeds,
            labels=labels
        )




In [ ]:
# 문제 10

# 다음 조건을 만족하도록 training step을 완성하시오.

# Vision Encoder freeze
# Projection + LM만 학습
# Loss는 language modeling loss 사용


# tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# models
vision_model = CLIPVisionModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

lm = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
lm.config.pad_token_id = lm.config.eos_token_id

model = MiniLLaVAModel(
    vision_model=vision_model,
    lm=lm,
    tokenizer=tokenizer
).to(device)

model.train()

# TODO 1: vision encoder freeze
for p in model.vision_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

# dummy input
batch_size = 1
pixel_values = torch.randn(batch_size, 3, 224, 224).to(device)

text = "Describe the image."
text_ids = tokenizer(
    text,
    return_tensors="pt"
).input_ids.to(device)

# vision token count
with torch.no_grad():
    v = model.vision_model(pixel_values=pixel_values)
num_vision_tokens = v.last_hidden_state.size(1)

# 🔑 labels: vision + first text token은 -100
labels = torch.cat(
    [
        torch.full(
            (batch_size, num_vision_tokens + 1),
            -100,
            device=device
        ),
        text_ids[:, 1:]
    ],
    dim=1
)

# forward
outputs = model(
    pixel_values=pixel_values,
    input_ids=text_ids,
    labels=labels
)

# TODO 2: loss 정의
loss = outputs.loss
loss.backward()
optimizer.step()

print("✅ Training step completed. Loss:", loss.item())